# 02B Portable Feature Ablation

This notebook runs controlled portable-model ablations on dataset `P` so you can compare which feature profile actually helps transfer.

In [7]:
from pathlib import Path
import pandas as pd
import subprocess
import sys

ROOT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling')
REPORTS_DIR = ROOT / 'outputs' / 'reports'
SCRIPT = ROOT / 'scripts' / 'train_and_save_models.py'

DATASET = 'P'
MODELS = ['XGBOOST']
FEATURE_PROFILES = [
    'lags_only',
    'lags_roll',
    'lags_roll_seasonal',
    'lags_roll_seasonal_category',
    'full',
]
TAG_PREFIX = 'portable_ablation'
HORIZONS = '1,2,3,4,5,6,7,8,9,10,11,12'


## Run Training

Run this cell once. It trains one tag per feature profile.

In [8]:
for profile in FEATURE_PROFILES:
    tag = f'{TAG_PREFIX}_{profile}'
    cmd = [
        sys.executable,
        str(SCRIPT),
        '--datasets', DATASET,
        '--skip-classical',
        '--boosting-models', *MODELS,
        '--feature-profile', profile,
        '--horizons', HORIZONS,
        '--tag', tag,
    ]
    print('RUNNING', ' '.join(cmd))
    subprocess.run(cmd, check=True)


## Compare Leaderboards

In [9]:
rows = []
for profile in FEATURE_PROFILES:
    tag = f'{TAG_PREFIX}_{profile}'
    path = REPORTS_DIR / f'{tag}_leaderboard.csv'
    if not path.exists():
        continue
    df = pd.read_csv(path)
    df['feature_profile'] = profile
    rows.append(df)

ablation = pd.concat(rows, ignore_index=True)
ablation[['feature_profile', 'dataset', 'model', 'WAPE', 'RMSE', 'Bias', 'MASE_mean']].sort_values(['WAPE', 'RMSE'])


,feature_profile,dataset,model,WAPE,RMSE,Bias,MASE_mean
4,full,P,XGBOOST,0.321966,3190.205051,167.418547,1.792354
3,lags_roll_seasonal_category,P,XGBOOST,0.334801,3310.416214,250.723306,1.676233
2,lags_roll_seasonal,P,XGBOOST,0.336065,3334.153750,255.419961,1.145124
1,lags_roll,P,XGBOOST,0.337404,3366.999768,243.229318,1.143137
0,lags_only,P,XGBOOST,0.359972,3515.198498,215.156699,1.374080


## Inspect One Profile

In [10]:
PROFILE_TO_VIEW = 'full'
tag = f'{TAG_PREFIX}_{PROFILE_TO_VIEW}'
pd.read_csv(REPORTS_DIR / f'{tag}_p_metrics.csv').head(30)


,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,P,XGBOOST,val,1,1728,0.315172,3303.923560,156.994658,1.943711,291.101186,0.355903
1,P,XGBOOST,val,2,1728,0.352564,3661.328997,306.592097,2.315707,325.637827,0.296296
2,P,XGBOOST,val,3,1728,0.321715,3156.494001,294.147638,2.139545,297.144363,0.248843
3,P,XGBOOST,val,4,1728,0.332828,3213.397151,313.549655,1.826120,307.409251,0.250579
4,P,XGBOOST,val,5,1728,0.349428,3267.923470,286.783460,1.167775,322.740738,0.310764
5,P,XGBOOST,val,6,1728,0.321637,3410.225252,70.539105,2.914859,297.072939,0.239583
6,P,XGBOOST,val,7,1728,0.344021,3409.152685,36.013478,1.631584,317.746889,0.431134
7,P,XGBOOST,val,8,1728,0.319517,3245.554007,113.414336,1.922804,295.114160,0.370949
8,P,XGBOOST,val,9,1728,0.319215,3355.046516,-112.886213,2.015915,294.835101,0.345486
9,P,XGBOOST,val,10,1728,0.326843,3440.183315,199.509048,2.316523,301.880749,0.285880
